# 📙 Module 04 · Notebook 02 — LLM untuk Produksi (di T4 gratis)

Model 7B tak muat `float16` di T4. Di sini kita pakai **teknik produksi** agar tetap jalan:
**4-bit quantization**, **memory profiling**, **batching**, dan **streaming**.

## Apa yang akan kita pelajari?
1. Kenapa 7B fp16 OOM di T4 (perhitungan)
2. **Quantization 4-bit** (bitsandbytes) → jalankan 7B di T4
3. **GPU memory profiling**
4. **Batching** untuk throughput
5. **Streaming** token
6. Jembatan ke RAG (Module 05)

In [1]:
# Pin transformers < 5 WAJIB: v5 punya regresi OOM bnb 4-bit (HF #43032).
!uv pip install -q "transformers>=4.44,<5" accelerate bitsandbytes

In [2]:
import torch, time
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TextStreamer, TextIteratorStreamer)

def gpu_mem(tag=""):
    a = torch.cuda.memory_allocated() / 1e9
    r = torch.cuda.memory_reserved() / 1e9
    print(f"[{tag}] allocated={a:.2f} GB | reserved={r:.2f} GB")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU: Tesla T4


## 1. Kenapa 7B `float16` OOM di T4

Bobot saja = **7.25 miliar × 2 byte ≈ 14.5 GB**, plus KV-cache & aktivasi ≈ **~16 GB** →
melebihi 16 GB T4. (Kita **tidak** memicu OOM sungguhan — itu bisa merusak runtime; cukup hitung.)

In [3]:
params_b = 7.25
for dtype, bytes_per in [("float16", 2), ("4-bit NF4", 0.5)]:
    gb = params_b * 1e9 * bytes_per / 1e9
    print(f"{dtype:10}: bobot ~{gb:.1f} GB  -> {'OOM di T4 (16GB)' if gb > 13 else 'muat di T4'}")

float16   : bobot ~14.5 GB  -> OOM di T4 (16GB)
4-bit NF4 : bobot ~3.6 GB  -> muat di T4


## 2. Quantization 4-bit (bitsandbytes)

NF4 menyimpan bobot dalam 4-bit (~1/4 memori). **Di T4 `bnb_4bit_compute_dtype` WAJIB
`torch.float16`** — `bfloat16` CRASH (T4 compute 7.5 < 8.0). Footprint ~5.5 GB → muat nyaman.

In [4]:
# Mistral-7B-Instruct-v0.3: Apache-2.0, non-gated.
# Jika repo Mistral minta akses, ganti ke alternatif non-gated berikut:
#   prod_model = "Qwen/Qwen2.5-7B-Instruct"   # Apache-2.0, 4-bit ~6.5 GB
prod_model = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # WAJIB fp16 di T4 (bf16 crash)
)
tok = AutoTokenizer.from_pretrained(prod_model)
m4 = AutoModelForCausalLM.from_pretrained(
    prod_model, quantization_config=bnb_config, device_map="auto",
)
gpu_mem("Mistral-7B 4-bit")

messages = [{"role": "user", "content": "Sebutkan 3 manfaat AI untuk UMKM, singkat."}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors="pt", return_dict=True).to(m4.device)
out = m4.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7,
                  pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[Mistral-7B 4-bit] allocated=4.14 GB | reserved=7.23 GB


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


1. Pengembangan produk: AI membantu UMKM memahami prefrensi konsumen lebih dalam dan membantu mengembangkan produk yang lebih sesuai dengan kebutuhan pasar.

2. Pengoptimalan bisnis: AI membantu UMKM mengoptimalkan bisnis dengan mengidentifikasi trend dan peluang bisnis, serta membantu merancang strategi bisnis yang lebih efektif.

3. Pemahaman konsumen: AI membantu UMKM memahami konsumen lebih dalam, antara lain dari langkah-langkah belanjaan, preferensi, sosial media, dan komentar pada produk.


## 3. GPU Memory Profiling

`torch.cuda.memory_allocated` (terpakai sekarang) vs `memory_reserved` (di-cache PyTorch) vs
`max_memory_allocated` (puncak). Higiene: `del model; torch.cuda.empty_cache()` saat ganti model.

In [5]:
print(f"peak allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
gpu_mem("sekarang")
# contoh higiene (jangan jalankan kalau masih butuh m4):
# del m4; torch.cuda.empty_cache(); gpu_mem("setelah dibebaskan")

peak allocated: 7.11 GB
[sekarang] allocated=4.15 GB | reserved=7.27 GB


## 4. Batching untuk Throughput

Memproses banyak prompt sekaligus jauh lebih efisien daripada satu-satu. Untuk decoder-only,
WAJIB `padding_side="left"` agar posisi generasi benar.

In [6]:
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

prompts = [
    "Terjemahkan ke Inggris: Selamat pagi.",
    "Apa kepanjangan dari CPU?",
    "Beri satu tips belajar coding.",
]
batch_msgs = [[{"role": "user", "content": p}] for p in prompts]
texts = [tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
         for m in batch_msgs]
enc = tok(texts, return_tensors="pt", padding=True).to(m4.device)

t0 = time.time()
out = m4.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
dt = time.time() - t0
for i, p in enumerate(prompts):
    gen = out[i][enc["input_ids"].shape[1]:]
    print(f"\nQ: {p}\nA: {tok.decode(gen, skip_special_tokens=True).strip()}")
print(f"\n{len(prompts)} prompt dalam 1 batch: {dt:.1f}s")


Q: Terjemahkan ke Inggris: Selamat pagi.
A: Good morning.

Q: Apa kepanjangan dari CPU?
A: CPU berarti Central Processing Unit. Dalam bahasa Inggris, CPU adalah singkatan dari Central Processing Unit. Dalam bahasa Indonesia, CPU disebut Unit Prosesor Pusat. Kata ini digunakan untuk mengac

Q: Beri satu tips belajar coding.
A: Saya akan memberi tips mengenai belajar coding yang dapat membantu Anda mencapai kemajuan yang lebih efisien.

1. **Belajar satu bahasa pemrograman se

3 prompt dalam 1 batch: 14.2s


## 5. Streaming Token

Daripada menunggu seluruh jawaban, **streaming** menampilkan token saat diproduksi (UX lebih
baik). `TextStreamer` mencetak ke output; `TextIteratorStreamer` (threaded) untuk aplikasi.

In [7]:
msg = [{"role": "user", "content": "Ceritakan singkat sejarah Transformer dalam NLP."}]
inp = tok.apply_chat_template(msg, add_generation_prompt=True,
                              return_tensors="pt", return_dict=True).to(m4.device)
streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)
_ = m4.generate(**inp, max_new_tokens=200, do_sample=True, temperature=0.7,
                pad_token_id=tok.eos_token_id, streamer=streamer)

Transformer adalah suatu model machine learning yang dikembangkan oleh Joshua Bengio, Giannii Vaswani, Akhil Akshitha, Llion Kaiser, Aidan N. Gomez, Mikhail Henry, Ilya Sutskever, Dani Yogatama, Jeff Dean, dan Geoffrey Hinton pada tahun 2017. Model ini terkenal dengan caranya memanfaatkan mekanisme self-attention (attention mechanism) dalam menginterpretasikan data masukan, yang menjadikannya lebih efisien daripada model sebelumnya seperti Recurrent Neural Network (RNN) dan Long Short-Term Memory (LSTM).

Transformer digunakan secara luas dalam bahasa pesan natur


In [8]:
from threading import Thread

it_streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
gen_kwargs = dict(**inp, max_new_tokens=120, do_sample=True, temperature=0.7,
                  pad_token_id=tok.eos_token_id, streamer=it_streamer)
Thread(target=m4.generate, kwargs=gen_kwargs).start()
for token in it_streamer:        # cocok untuk app/web: konsumsi token saat datang
    print(token, end="", flush=True)
print()

Transformer adalah metode pemrosesan bahasa alamat (NLP) yang diperkenalkan oleh Vaswani dan kollega dalam tahun 2017 dengan artikel "Attention is All You Need". Ini adalah metode yang mencapai hasil yang signifikan dalam beberapa tugas NLP, seperti pengolahan teks, traduisi teks, dan prediksi teks.

Transformer meng


## Ringkasan & Jembatan ke RAG (Module 05)

| Teknik | Gunanya |
|--------|---------|
| Quantization 4-bit | jalankan 7B di GPU kecil |
| Memory profiling | ukur & jaga footprint |
| Batching | throughput lebih tinggi |
| Streaming | UX responsif |

Tapi model tetap bisa **berhalusinasi** dan punya **knowledge cutoff** — ia tak tahu data
terbaru/privat kita. **Module 05 (RAG)** memberi LLM "buku contekan": mengambil dokumen relevan
lalu menyuntikkannya ke prompt agar jawaban akurat & terkini.